# 1 Exploratory analysis — Berlin & Vienna
**Team Shtrudel · Region: Germany & Austria (EUR)** · Training cities: **A = Berlin**, **B = Vienna** · City C (Munich) is deliberately **not** loaded here.

## Summary

**What we know**

* Each row is one listing as it looked on the snapshot date. Each listing appears once.
* About a quarter to a third of listings have no price. These are almost always inactive listings that cannot be booked, so the gap is not random. We keep them in the data but do not train on them.
* Prices are in euros, even though the file shows them with a dollar sign.
* Most listings are moderately priced, with a few extremely expensive ones. We model the logarithm of price and flag outliers within each room type instead of using one fixed cut-off.
* **Leakage:** the estimated revenue column is calculated directly from the price, and the price-quote columns repeat the price. All of them are excluded from the model.
* **The two cities differ:** a large share of Berlin listings are long-stay only (a result of local short-term rental rules), which is rare in Vienna. The cities also describe neighbourhoods at different levels of detail, some columns are filled in only one city, and some Vienna district names arrive with broken characters.
* **Main price drivers:** listing size (number of guests, bedrooms, beds) and location (central districts are about twice as expensive as outer ones). Long-stay listings are much cheaper per night. Reviews and host size matter little.
* Several host-profile columns are empty in both cities in this snapshot, so the pipeline must not rely on them.

**What we do not know yet**

* Whether the price on the snapshot date is typical for the listing: it is a single quote for specific dates, and some quotes are derived from monthly prices.
* Whether the very high prices belong to real luxury listings or are placeholders.
* How a new city's data and price level will differ. That is what the city C test will show.

## 0 Setup
Set path to the folder that holds one `listings.csv.gz` per city

In [0]:
import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# "/Volumes/workspace/airbnb/raw"  -> <BASE_PATH>/<city>/listings.csv.gz
BASE_PATH = os.environ.get("AIRBNB_RAW", "/Volumes/workspace/airbnb/raw")

CITIES = {
    "berlin": "2026-09-19",
    "vienna": "2026-09-17",
}
RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

In [0]:
def find_file(city):
    for name in ("listings.csv.gz", "listings.csv"):
        p = os.path.join(BASE_PATH, city, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No listings file for {city} under {BASE_PATH}/{city}/")

def money(s):
    """'$1,234.00' -> 1234.0 (value is in local currency, EUR here)."""
    return pd.to_numeric(s.astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")

frames = []
for city, snap in CITIES.items():
    d = pd.read_csv(find_file(city), low_memory=False)
    d.insert(0, "city", city)
    d.insert(1, "snapshot_date", snap)
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)
raw["price_eur"] = money(raw["price"])
raw.shape

## 1 · The dataset
Snapshot dates, grain and record counts.

In [0]:
summary = raw.groupby("city").agg(
    snapshot_date=("snapshot_date", "first"),
    rows=("id", "size"),
    unique_ids=("id", "nunique"),
    last_scraped=("last_scraped", lambda s: ", ".join(sorted(s.dropna().unique()))),
    hosts=("host_id", "nunique"),
    with_price=("price_eur", "count"),
)
summary["share_with_price"] = (summary.with_price / summary.rows).round(3)
summary

**Grain:** one row = one listing on the snapshot date. Note that `last_scraped` can be a day or two later than the snapshot folder date (Vienna folder 2026-09-17, scraped 2026-09-19); we report the folder date.

## 2 · Column profile
### 2.1 Schema differences between the cities

In [0]:
miss = raw.groupby("city").apply(lambda g: g.isna().mean(), include_groups=False).T.round(3)
fully_empty = miss[(miss == 1).any(axis=1)]
print("Columns 100% empty in at least one city:")
fully_empty

Two kinds of empty columns:

* **Empty in both cities** (`host_since`, `host_response_rate`, `host_acceptance_rate`, `host_verifications`, `bathrooms`, `instant_bookable`, `neighbourhood`, `calendar_updated`…): dropped by this version of the Inside Airbnb export. The pipeline must not reference them, and should **not fail** if they come back. `bathrooms` is recoverable from `bathrooms_text`.
* **Empty in one city only** — `neighbourhood_group_cleansed` and `license` exist in Berlin, empty in Vienna. Any feature built on them would not transfer. This is exactly the kind of difference we expect in city C.

### 2.2 Columns we use: types and missingness

In [0]:
USE = ["price_eur", "room_type", "property_type", "accommodates", "bedrooms", "beds", "bathrooms_text",
       "neighbourhood_cleansed", "latitude", "longitude", "minimum_nights", "availability_30",
       "availability_365", "number_of_reviews", "number_of_reviews_ltm", "review_scores_rating",
       "review_scores_location", "reviews_per_month", "host_is_superhost", "calculated_host_listings_count",
       "amenities", "has_availability"]
prof = pd.DataFrame({
    "dtype_raw": raw[USE].dtypes.astype(str),
    "missing_berlin": raw.loc[raw.city == "berlin", USE].isna().mean().round(3),
    "missing_vienna": raw.loc[raw.city == "vienna", USE].isna().mean().round(3),
    "n_unique": raw[USE].nunique(),
})
prof

Type issues the pipeline must fix: `price` is text with a currency sign; `host_is_superhost` / `has_availability` are `'t'/'f'` strings; `bathrooms_text` mixes number and kind (`"1.5 shared baths"`, `"Half-bath"`); `amenities` is a JSON list stored as text; dates are strings.

### 2.3 Missingness pattern

In [0]:
raw["inactive"] = raw["availability_365"].eq(0)
raw["no_reviews_ltm"] = raw["number_of_reviews_ltm"].eq(0)
pattern = raw.groupby(["city", "inactive"]).agg(rows=("id", "size"),
                                                price_missing=("price_eur", lambda s: s.isna().mean()))
pattern.round(3)

In [0]:
pd.crosstab([raw.city, raw.price_eur.isna().rename("price_missing")],
            raw.review_scores_rating.isna().rename("rating_missing"), normalize="index").round(3)

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

prof[["missing_berlin", "missing_vienna"]].query("missing_berlin > 0 or missing_vienna > 0") \
    .sort_values("missing_berlin").plot.barh(ax=ax[0])
ax[0].set_title("Share of missing values (columns we use)"); ax[0].set_xlabel("share missing")

avail_bin = pd.cut(raw.availability_365, [-1, 0, 30, 90, 180, 270, 365],
                   labels=["0", "1-30", "31-90", "91-180", "181-270", "271-365"])
raw.groupby([avail_bin, "city"], observed=True).price_eur.apply(lambda s: s.isna().mean()) \
    .unstack().plot.bar(ax=ax[1], rot=0)
ax[1].set_title("Price missing by availability_365 (days open in next year)"); ax[1].set_ylabel("share without price")
plt.tight_layout(); plt.show()

* **Missing prices are not random.** Almost all listings that are closed for the whole year (96–98%) have no price. Among active listings only 6–10% are missing it. A missing price means the listing could not be booked on the snapshot date, not that the data is broken. We do not fill in the price; these listings are left out of training but kept in the data and shown on the dashboard.
* **Review scores are missing for listings with no reviews yet** (about 15–18%). For the model we fill them with the median and add a flag that shows whether the listing has reviews.
* **Beds and bedrooms** are missing in very few listings (2–3% and under 0.1%). We fill them based on the number of guests.

## 3 · Distributions

In [0]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for city, g in raw.groupby("city"):
    g.price_eur.dropna().clip(upper=1000).plot.hist(bins=80, alpha=.5, ax=ax[0], label=city)
    np.log10(g.price_eur.dropna()).plot.hist(bins=80, alpha=.5, ax=ax[1], label=city)
    g.availability_365.plot.hist(bins=50, alpha=.5, ax=ax[2], label=city)
ax[0].set_title("Price, EUR (clipped at 1000)"); ax[1].set_title("log10(price)"); ax[2].set_title("availability_365")
for a in ax: a.legend()
plt.tight_layout(); plt.show()

In [0]:
raw.groupby("city").price_eur.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1)

In [0]:
raw.pivot_table(index="room_type", columns="city", values="price_eur", aggfunc=["count", "median"]).round(0)

* **Price:** most listings are moderately priced, but a few are very expensive. After taking the logarithm, the distribution looks much more even, so we predict the log of the price.
* **Price level:** typical prices are almost the same in both cities (about €125–130). For entire homes Berlin is a bit more expensive (€155 vs €131).
* **Availability:** most listings are either closed for the whole year or open almost every day, with few in between. We show availability on the dashboard but do not predict it.
* **Room types:** Vienna has more entire homes (83%) than Berlin (68%). Shared and hotel rooms are rare (under 1%), too few for the model to learn them separately.

### 3.1 Minimum nights

In [0]:
bins = [0, 1, 2, 3, 6, 29, 89, 10**6]
labels = ["1", "2", "3", "4-6", "7-29", "30-89", "90+"]
mn = pd.crosstab(raw.city, pd.cut(raw.minimum_nights, bins=bins, labels=labels), normalize="index").round(3)
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
mn.T.plot.bar(ax=ax[0], rot=0); ax[0].set_title("Minimum nights required"); ax[0].set_ylabel("share of listings")
raw.assign(is_long_stay=raw.minimum_nights >= 30).groupby(["city", "is_long_stay"]).price_eur.median() \
    .unstack().plot.bar(ax=ax[1], rot=0)
ax[1].set_title("Median price: short stay vs long stay (min nights ≥ 30)"); ax[1].set_ylabel("EUR")
plt.tight_layout(); plt.show()
mn

**31% of Berlin listings require a stay of at least 90 nights**, compared with under 1% in Vienna. Berlin restricts short-term rentals, so many hosts offer their places for longer stays instead. For these listings the nightly price is derived from a monthly price, so they are a different kind of product.

We keep them, but give the model a flag for long-stay listings (minimum stay of 30 nights or more) and report the error for this group separately. The right chart shows why this matters: long-stay listings cost about **half as much per night** (Berlin €66 vs €149, Vienna €71 vs €130).

## 4 · Outliers and how we classify them

In [0]:
priced = raw[raw.price_eur.notna()].copy()
priced["log_p"] = np.log(priced.price_eur)
grp = priced.groupby(["city", "room_type"]).log_p
q1, q3 = grp.transform(lambda s: s.quantile(.25)), grp.transform(lambda s: s.quantile(.75))
iqr = q3 - q1
priced["outlier"] = np.select(
    [priced.price_eur < 15,
     priced.log_p > q3 + 3 * iqr,
     priced.log_p < q1 - 3 * iqr],
    ["implausible_low", "extreme_high", "extreme_low"], default="ok")
pd.crosstab(priced.city, priced.outlier)

In [0]:
order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, (city, g) in zip(axes, priced.groupby("city")):
    ok = g[g.outlier == "ok"]; out = g[g.outlier != "ok"]
    ax.boxplot([ok.loc[ok.room_type == r, "price_eur"] for r in order], showfliers=False)
    ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order)
    xs = out.room_type.map({r: i + 1 for i, r in enumerate(order)})
    ax.scatter(xs, out.price_eur, color="crimson", s=14, label="flagged outlier")
    ax.set_yscale("log"); ax.set_title(f"{city}: price by room type (log scale)"); ax.legend()
axes[0].set_ylabel("price, EUR")
plt.tight_layout(); plt.show()

In [0]:
cols = ["city", "price_eur", "room_type", "accommodates", "bedrooms", "minimum_nights",
        "availability_365", "number_of_reviews", "outlier"]
priced.loc[priced.outlier != "ok", cols].sort_values("price_eur", ascending=False).head(15)

Classification (applied per city × room type, on log price, so a €900 villa for 12 is not treated like a €900 studio):

| Class | Rule | What it usually is | Treatment |
|---|---|---|---|
| `implausible_low` | price < €15 | placeholder / wrong unit | drop from training |
| `extreme_high` | log price > Q3 + 3·IQR | placeholder prices (€9,999), event pricing, or real luxury homes | drop from training, keep in silver & dashboard |
| `extreme_low` | log price < Q1 − 3·IQR | data errors, monthly price divided oddly | drop from training |
| `ok` | otherwise | — | use |

High prices on listings with 0 reviews and 365 availability are most suspicious (never booked at that price). The rule is relative, so it also works for a city with a different price level.

## 4b · Text encoding in Vienna

In [0]:
def fix_mojibake(s):
    """Vienna names were decoded with the wrong code page (Mac Roman read as Latin-1)."""
    if isinstance(s, str) and any(("\x80" <= ch <= "\x9f") or ch == "§" for ch in s):
        try:
            return s.encode("latin-1").decode("mac_roman")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return s
    return s

bad = sorted({s for s in raw.neighbourhood_cleansed.unique() if fix_mojibake(s) != s})
priced["neighbourhood_cleansed"] = priced.neighbourhood_cleansed.map(fix_mojibake)
pd.DataFrame({"raw": [repr(s) for s in bad], "fixed": [fix_mojibake(s) for s in bad]})

Four Vienna districts (`Döbling`, `Währing`, `Landstraße`, `Rudolfsheim-Fünfhaus`) arrive with broken characters. Berlin names are correct. The pipeline applies `fix_mojibake` only to strings that contain the tell-tale characters, so correct names in other cities are left untouched.

## 4c · What drives price — input for the model
Only training-safe columns (see section 6). Outliers excluded.

In [0]:
clean = priced[priced.outlier == "ok"].copy()
clean["n_amenities"] = clean.amenities.apply(lambda s: len(json.loads(s)) if isinstance(s, str) else np.nan)
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
clean[clean.accommodates <= 10].groupby(["accommodates", "city"]).price_eur.median().unstack().plot(marker="o", ax=ax[0])
ax[0].set_title("Median price by number of guests"); ax[0].set_ylabel("EUR")
cand = ["accommodates", "bedrooms", "beds", "n_amenities", "minimum_nights", "number_of_reviews",
        "reviews_per_month", "review_scores_rating", "review_scores_location",
        "calculated_host_listings_count", "availability_365"]
corr = clean.groupby("city").apply(lambda g: g[cand].corrwith(g.log_p, method="spearman"), include_groups=False).T
corr.sort_values("berlin").plot.barh(ax=ax[1]); ax[1].axvline(0, color="grey", lw=.8)
ax[1].set_title("Spearman correlation with log price")
plt.tight_layout(); plt.show()
corr.round(2)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (city, g) in zip(axes, clean.groupby("city")):
    sc = ax.scatter(g.longitude, g.latitude, c=g.log_p, s=3, cmap="viridis",
                    vmin=np.log(40), vmax=np.log(400))
    ax.set_title(f"{city}: listings coloured by log price"); ax.set_aspect(1.5); ax.set_xticks([]); ax.set_yticks([])
cb = fig.colorbar(sc, ax=axes, shrink=.8); cb.set_ticks(np.log([50, 100, 200, 400])); cb.set_ticklabels(["€50", "€100", "€200", "€400"])
plt.show()

In [0]:
nb = clean.groupby(["city", "neighbourhood_cleansed"]).price_eur.agg(median="median", listings="size").reset_index()
nb = nb[nb.listings >= 30]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (city, g) in zip(axes, nb.groupby("city")):
    g = g.sort_values("median"); g = pd.concat([g.head(8), g.tail(8)])
    ax.barh(g.neighbourhood_cleansed, g["median"], color=["#4c72b0"] * 8 + ["#dd8452"] * 8)
    ax.set_title(f"{city}: cheapest and most expensive neighbourhoods (≥ 30 listings)"); ax.set_xlabel("median price, EUR")
plt.tight_layout(); plt.show()

* **Size matters most** in both cities: the more guests a listing takes (and the more bedrooms and beds it has), the higher the price. Price rises faster with size in Berlin than in Vienna, so the model also needs to know which city a listing is in.
* **Location matters:** central districts are clearly more expensive. In Vienna the old town costs about €227 against about €110 in outer districts; in Berlin central areas cost about €190 against about €80. District names are different in every city and will not exist in city C, so features based on coordinates, such as distance to the centre, will transfer better. The map also shows that Vienna has one expensive centre, while Berlin has several. Distance to the centre will therefore work better for Vienna than for Berlin.
* **Longer minimum stays mean lower prices** (see the long-stay section above).
* **Reviews and the number of listings a host has matter little.** They can be used as secondary features at most.
* **The number of amenities** matters more in Berlin than in Vienna.

## 5 · Cleaning & transformations the pipeline must apply

| # | Step | Reason |
|---|---|---|
| 1 | Add the city name and snapshot date to every row | Data from several cities is stored in one table |
| 2 | Turn the price from text into a number | It is stored as text with a dollar sign, but the value is in euros |
| 3 | Keep listings without a price, but mark them and leave them out of training | They are inactive listings; we never invent the price we want to predict |
| 4 | Mark price outliers separately for each city and room type; leave them out of training | A fixed price limit would not work for another city |
| 5 | Turn "t"/"f" values into true/false | Clean, correctly typed data |
| 6 | Get the number of bathrooms from the bathroom description, and note whether it is shared | The numeric bathrooms column is empty |
| 7 | Turn the amenities list into a count plus a few yes/no flags (e.g. kitchen, washer, air conditioning) | The list is stored as text and cannot be used directly |
| 8 | Fix broken characters in district names | Four Vienna districts are spelled with wrong characters |
| 9 | Turn date text into real dates | Clean, correctly typed data |
| 10 | Select columns by name and do not fail if one is missing | Columns differ between cities and snapshots |
| 11 | Remove leakage columns before building model features | They reveal the price |
| 12 | Add a long-stay flag (minimum stay of 30 nights or more) | Long-stay listings are priced very differently |
| 13 | Fill missing beds and bedrooms from the number of guests; fill missing review scores with the median and add a "has reviews" flag | Only a few values are missing, and for a clear reason |

## 6 · Leakage review
Columns that must **not** be used to predict price.

In [0]:
p = raw.price_eur
checks = pd.DataFrame({
    "price_quote_price_per_night == price": ((money(raw.price_quote_price_per_night) - p).abs() < 0.01)[p.notna() & raw.price_quote_price_per_night.notna()].groupby(raw.city).mean(),
    "estimated_revenue_l365d == price × occupancy": ((p * raw.estimated_occupancy_l365d - money(raw.estimated_revenue_l365d)).abs() < 1)[p.notna()].groupby(raw.city).mean(),
}).round(3)
checks

In [0]:
json.loads(raw.price_quote_raw.dropna().iloc[0])["quote"] | {"raw_price_line_items": "..."}

| Column(s) | Why it must be excluded |
|---|---|
| `estimated_revenue_l365d` | **Calculated from the price:** it equals price × estimated occupancy for every listing that has a price |
| `price_quote_price_per_night` | The same number as the price |
| `price_quote_total_price`, `price_quote_raw` | Contain the price for a sample stay, so they reveal the price directly |
| `price_quote_checkin_date`, `price_quote_checkout_date` | Filled only when a price exists, so they reveal whether the listing has a price |
| `estimated_occupancy_l365d` | Depends on the price (cheaper places get booked more), and is not known for a new listing |
| `availability_30/60/90` | Partly depend on the price: cheap listings get booked and show less free time. We keep only yearly availability as a sign of how active a listing is, and check the model with and without it |
| `id`, `host_id`, `scrape_id`, links, `name`, `description` | Identifiers and free text: the model would memorise specific listings, which does not help in a new city |
| `calendar_last_scraped`, `last_scraped`, `source` | Technical information about data collection, not about the listing |

**Allowed features** (known for any listing and not based on its price): room and property type, number of guests, bedrooms, beds, bathrooms, location, minimum stay and the long-stay flag, amenities, reviews, superhost status, and how many listings the host has.